[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/03_Training_Strategies/02_pretraining_objectives.ipynb)

# 02. Pretraining Objectives for Multimodal Models

**Beyond contrastive learning** — modern models use multiple objectives.

**This notebook covers:**
- ITC (Image-Text Contrastive) — what CLIP uses
- ITM (Image-Text Matching) — binary match/no-match
- MLM (Masked Language Modeling) — predict masked words
- Image-Grounded Text Generation — caption objective
- How BLIP/BLIP-2 combines all of these

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/03_Training_Strategies")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *

set_style()

In [ ]:
# Overview: All pretraining objectives in one diagram

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Multimodal Pretraining Objectives', fontsize=18, fontweight='bold')

# 1. ITC
ax = axes[0, 0]
ax.set_xlim(0, 8)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('1. ITC (Image-Text Contrastive)', fontsize=13, fontweight='bold', color='#E74C3C')
draw_architecture_block(ax, 2, 4.5, 2.5, 0.7, 'Image Enc', '#E74C3C')
draw_architecture_block(ax, 6, 4.5, 2.5, 0.7, 'Text Enc', '#3498DB')
draw_architecture_block(ax, 4, 2.5, 4, 1, 'Contrastive Loss\n(InfoNCE)', '#9B59B6')
draw_arrow(ax, (2, 4.0), (3, 3.1))
draw_arrow(ax, (6, 4.0), (5, 3.1))
ax.text(4, 1, 'Pull matching pairs together\nPush non-matching apart', 
        ha='center', fontsize=9, style='italic')

# 2. ITM
ax = axes[0, 1]
ax.set_xlim(0, 8)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('2. ITM (Image-Text Matching)', fontsize=13, fontweight='bold', color='#3498DB')
draw_architecture_block(ax, 2, 4.5, 2.5, 0.7, 'Image Feat', '#E74C3C')
draw_architecture_block(ax, 6, 4.5, 2.5, 0.7, 'Text Feat', '#3498DB')
draw_architecture_block(ax, 4, 3, 4, 0.7, 'Cross-Attention', '#F39C12')
draw_architecture_block(ax, 4, 1.5, 3, 0.7, 'Match? Yes/No', '#2ECC71')
draw_arrow(ax, (2, 4.0), (3, 3.4))
draw_arrow(ax, (6, 4.0), (5, 3.4))
draw_arrow(ax, (4, 2.5), (4, 2.0))
ax.text(4, 0.7, 'Binary classification:\nDo image and text match?', 
        ha='center', fontsize=9, style='italic')

# 3. MLM
ax = axes[1, 0]
ax.set_xlim(0, 8)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('3. MLM (Masked Language Modeling)', fontsize=13, fontweight='bold', color='#2ECC71')
draw_architecture_block(ax, 2, 4.5, 2.5, 0.7, 'Image Feat', '#E74C3C')
draw_architecture_block(ax, 6, 4.5, 2.5, 0.7, 'a [MASK] cat', '#3498DB')
draw_architecture_block(ax, 4, 3, 4, 0.7, 'Cross-Attention', '#F39C12')
draw_architecture_block(ax, 4, 1.5, 3, 0.7, 'Predict: "cute"', '#2ECC71')
draw_arrow(ax, (2, 4.0), (3, 3.4))
draw_arrow(ax, (6, 4.0), (5, 3.4))
draw_arrow(ax, (4, 2.5), (4, 2.0))
ax.text(4, 0.7, 'Predict masked words\nusing image as context', 
        ha='center', fontsize=9, style='italic')

# 4. Generation
ax = axes[1, 1]
ax.set_xlim(0, 8)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('4. Image-Grounded Generation', fontsize=13, fontweight='bold', color='#F39C12')
draw_architecture_block(ax, 2, 4.5, 2.5, 0.7, 'Image Feat', '#E74C3C')
draw_architecture_block(ax, 6, 4.5, 2.5, 0.7, '[BOS] a cute', '#3498DB')
draw_architecture_block(ax, 4, 3, 4, 0.7, 'Causal Decoder', '#F39C12')
draw_architecture_block(ax, 4, 1.5, 3, 0.7, 'Next: "cat"', '#2ECC71')
draw_arrow(ax, (2, 4.0), (3, 3.4))
draw_arrow(ax, (6, 4.0), (5, 3.4))
draw_arrow(ax, (4, 2.5), (4, 2.0))
ax.text(4, 0.7, 'Autoregressive text generation\nconditioned on image', 
        ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.savefig('../assets/pretraining_objectives.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Implement each loss function

def itc_loss(img_emb, txt_emb, temperature=0.07):
    """Image-Text Contrastive (same as CLIP)."""
    img_emb = F.normalize(img_emb, dim=-1)
    txt_emb = F.normalize(txt_emb, dim=-1)
    logits = img_emb @ txt_emb.T / temperature
    labels = torch.arange(len(img_emb), device=img_emb.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2


def itm_loss(fused_features, is_matched):
    """Image-Text Matching: binary classification."""
    classifier = nn.Linear(fused_features.shape[-1], 2)
    logits = classifier(fused_features)
    return F.cross_entropy(logits, is_matched.long())


def mlm_loss(predicted_logits, target_ids, mask_positions):
    """Masked Language Modeling."""
    # Only compute loss at masked positions
    masked_logits = predicted_logits[mask_positions]
    masked_targets = target_ids[mask_positions]
    return F.cross_entropy(masked_logits, masked_targets)


def generation_loss(predicted_logits, target_ids):
    """Autoregressive caption generation loss."""
    # predicted_logits: [B, T, vocab_size]
    # target_ids: [B, T] (shifted right)
    B, T, V = predicted_logits.shape
    return F.cross_entropy(
        predicted_logits.view(B*T, V),
        target_ids.view(B*T)
    )


# Demo each loss
B, D, V = 4, 128, 1000

img_emb = torch.randn(B, D)
txt_emb = torch.randn(B, D)

print("Loss values (random model, before training):")
print(f"  ITC:        {itc_loss(img_emb, txt_emb).item():.4f}")

fused = torch.randn(B, D)
matched = torch.tensor([1, 0, 1, 0])  # 2 matching, 2 non-matching
print(f"  ITM:        {itm_loss(fused, matched).item():.4f}")

pred_logits = torch.randn(B, 10, V)  # 10 tokens
target = torch.randint(0, V, (B, 10))
print(f"  Generation: {generation_loss(pred_logits, target).item():.4f}")

In [ ]:
# How BLIP combines objectives (visual)

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('BLIP: Multi-Objective Pretraining', fontsize=18, fontweight='bold', pad=20)

# Shared image encoder
draw_architecture_block(ax, 2, 6.5, 3, 0.8, 'Image Encoder\n(shared, frozen later)', '#E74C3C')

# Three text modes
draw_architecture_block(ax, 7, 7, 2.5, 0.7, 'Unimodal\nText Encoder', '#3498DB')
draw_architecture_block(ax, 10, 7, 2.5, 0.7, 'Image-Grounded\nText Encoder', '#F39C12')
draw_architecture_block(ax, 13, 7, 2.5, 0.7, 'Image-Grounded\nText Decoder', '#2ECC71')

# Losses
draw_architecture_block(ax, 4.5, 4.5, 3, 0.7, 'ITC Loss', '#9B59B6')
draw_architecture_block(ax, 8.5, 4.5, 3, 0.7, 'ITM Loss', '#9B59B6')
draw_architecture_block(ax, 12.5, 4.5, 3, 0.7, 'LM Loss', '#9B59B6')

draw_architecture_block(ax, 8.5, 2, 8, 1, 'Total Loss = α·ITC + β·ITM + γ·LM', '#34495E', fontsize=12)

draw_arrow(ax, (2, 6.0), (3.5, 5.0))
draw_arrow(ax, (7, 6.5), (5, 5.0))
draw_arrow(ax, (2, 6.0), (7.5, 5.0))
draw_arrow(ax, (10, 6.5), (9, 5.0))
draw_arrow(ax, (2, 6.0), (11.5, 5.0))
draw_arrow(ax, (13, 6.5), (13, 5.0))

for x in [4.5, 8.5, 12.5]:
    draw_arrow(ax, (x, 4.0), (x, 2.7))

plt.tight_layout()
plt.savefig('../assets/blip_objectives.png', dpi=150, bbox_inches='tight')
plt.show()

## Objective Comparison Table

| Objective | What it Learns | Used In | Compute Cost |
|-----------|---------------|---------|------|
| **ITC** | Global alignment (image↔text) | CLIP, BLIP | Low |
| **ITM** | Fine-grained matching | BLIP, ALBEF | Medium |
| **MLM** | Language understanding + grounding | BLIP, BEiT-3 | Medium |
| **Generation** | Caption/answer generation | BLIP, BLIP-2 | High |
| **ITC + ITM + Gen** | All abilities | BLIP | High (but most capable) |

**For low compute:** Start with ITC only (CLIP-style), add ITM if needed.

---
**Next:** `03_training_pipeline.ipynb` - Full training pipeline from scratch